# Engineering notebook: MNIST autoencoder compression

Written before scaffolding `mnist-ae-compression-xyzh`.

This notebook is the design record. The repository layout, stage scripts, and task-analysis notebook come after it and follow the decisions below. It does not import a package, because that package does not exist yet.


## Task

Train a simple autoencoder on MNIST. The network compresses each digit into a short code and reconstructs the image from that code. Optuna compares latent sizes. The result to keep is reconstruction error against latent size, plus a grid of original images next to their reconstructions.

A later notebook, `notebooks/task_analysis.ipynb`, reads the finished study and checkpoint and draws those two views. It is not a second design pass.


## What gets rolled from the template

Source template: `dl-model-template-smbo`. There is no Copier or Cookiecutter step. A rolled repo keeps the six-stage layout:

1. `01_get_data` reads a mock SQL file and loads images from torchvision.
2. `02_engineer` applies the data contract and writes the modeling table plus a pixel tensor.
3. `03_split` keeps the official train and test sides.
4. `04_tune` runs Optuna on the train split only and writes the four Optuna HTML plots plus `reports/mnist_ae_compression/best_params.json`.
5. `05_train` fits the chosen network, writes a loss curve with a train series and a test series, and writes `reports/mnist_ae_compression/reconstructions/epoch_XXXX.png` for a fixed set of training digits.
6. `06_eval` scores both splits from the checkpoint.

Contracts stay grouped as `key`, `target`, `features`, and `log`. The pixel tensor stays `storage: npy`, aligned to the parquet row order. The DVC remote stays the local `demo` remote on `dvc-remote/`. Checkpoint load stays local-file first, then `dvc.api.open`.

The template network is a CIFAR-10 classifier (cross-entropy, accuracy, macro F1, ROC AUC, confusion matrices). Those scores do not measure compression. The rolled reporting rule replaces them with reconstruction MSE, MAE, compression ratio, the latent-size curve, and reconstruction grids. The digit label remains the contract target so the official split and stratified folds still have a class column. The loss does not use that label.


## Reference, and what is left out

Shape of the network comes from `VAE` in `VAE_Tutorial.ipynb`: two Linear-BatchNorm-ReLU-Dropout blocks, a Linear down to `hidden_dim // 2`, a mirrored decoder, and Xavier initialization.

That notebook is a variational autoencoder on 2D moons. This project is a vanilla autoencoder on MNIST. These pieces are not copied:

- `fc_mu` and `fc_logvar`
- `reparameterize`
- the KL term and the beta warmup
- sampling `z` from a standard normal to invent new points
- `ReduceLROnPlateau`

The bottleneck is one linear layer, `fc_z`, from `hidden_dim // 2` to `latent_dim`. `encode` returns that code. `decode` maps it back. `forward` returns only the reconstruction.

Pixels follow the template, not the moons demo. Stage 01 stores uint8 NHWC. Stage 02 divides by 255, subtracts 0.5, divides by 0.5, and stores float32 NCHW in [-1, 1]. MNIST is one channel, shape `[1, 28, 28]`, so the flat input length is 784. The decoder's last layer is `Tanh` so a reconstruction stays in that same range. The moons decoder has no output activation because those coordinates are not scaled to [-1, 1].


## Network specification

The next cell is the network the package class `Autoencoder` must match. Training, Optuna, and checkpoint IO live in the stage scripts, not here. Dropout is a constructor argument so the search can move it; the tutorial value 0.1 is the fixed setting on the latent-size trials.


In [ ]:
import torch
from torch import nn


class Autoencoder(nn.Module):
    """Deterministic autoencoder. The bottleneck is one linear code, not a Gaussian."""

    def __init__(self, input_dim: int, latent_dim: int, hidden_dim: int, dropout: float = 0.1) -> None:
        super().__init__()
        if latent_dim < 1:
            raise ValueError("latent_dim must be at least 1")
        if hidden_dim < 2:
            raise ValueError("hidden_dim must be at least 2")
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        self.hidden_dim = hidden_dim
        half = hidden_dim // 2
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, half),
            nn.BatchNorm1d(half),
            nn.ReLU(),
        )
        self.fc_z = nn.Linear(half, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, half),
            nn.BatchNorm1d(half),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(half, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, input_dim),
            nn.Tanh(),
        )
        self._initialize_weights()

    def _initialize_weights(self) -> None:
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.constant_(module.bias, 0)
            elif isinstance(module, nn.BatchNorm1d):
                nn.init.constant_(module.weight, 1)
                nn.init.constant_(module.bias, 0)

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc_z(self.encoder(x))

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        return self.decoder(z)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        flat = x.reshape(x.shape[0], -1)
        reconstructed = self.decode(self.encode(flat))
        return reconstructed.reshape_as(x)


probe = Autoencoder(input_dim=784, latent_dim=16, hidden_dim=256)
batch = torch.zeros(4, 1, 28, 28)
restored = probe(batch)
code = probe.encode(batch.reshape(4, -1))
assert restored.shape == batch.shape
assert code.shape == (4, 16)
assert not hasattr(probe, "fc_mu")
assert not hasattr(probe, "reparameterize")


## Loss and training

The loss is mean squared error between the normalized image and the reconstruction. There is no KL term.

The optimizer is Adam. Learning rate, weight decay, and batch size are hyperparameters. Each fit starts by resetting modules and applying Xavier again, so a trial does not keep another trial's weights. Gradients are clipped to global norm 1, which is the stability step from the tutorial loop. The beta schedule and the plateau scheduler are not used.

Tune and train both score a held-out loader in eval mode. That score is recorded. It does not stop the epoch loop and it does not choose the checkpoint. Hyperparameters are chosen only from cross-validation on the train split.


## Latent size, compression, and error

A latent size is the length of `z`. The image has 784 values, so the compression ratio is `784 / latent_dim`. Ratio 392 means a code of length 2. Ratio 6.125 means a code of length 128.

Reconstruction error is the MSE on the normalized pixels. Compression error in this project is that same MSE: it is the information the code failed to keep. A shorter code is a tighter bottleneck, so the error curve should fall as the latent size grows, with the steep part at the smallest sizes where a digit no longer fits.

Sizes in the study: 2, 4, 8, 16, 32, 64, 128.


## Optuna

Study name `mnist_ae`, storage `reports/mnist_ae_compression/mnist_ae.db`, direction minimize. The objective is the mean validation MSE across stratified folds of the train split. Folds are stratified on the digit so each fold still contains every class. The test images are not read.

When the study is empty, enqueue one trial per latent size, up to `n_trials`, with the other settings fixed: hidden width 256, dropout 0.1, learning rate `1e-3`, weight decay `1e-6`, batch size 128. Those fixed trials are the comparison the error curve needs. If `n_trials` is larger than the size list, further trials use TPE over latent size, hidden width in {128, 256, 512}, dropout in [0, 0.3], learning rate, weight decay, and batch size in {64, 128, 256}.

A finished study still writes the template Optuna HTML plots: optimization history, parameter importances, parallel coordinate, and parameter slice. It also writes `reports/mnist_ae_compression/plots/reconstruction_error_vs_latent.png`. Points that share a latent size are averaged.

`reports/mnist_ae_compression/best_params.json` stores `params`, `val_mse`, `n_trials`, `epochs`, and `n_splits`. `val_mse` is the study best value. Demo defaults match the template's short run shape: enough trials to cover the seven sizes, 2 epochs inside the search, then 8 epochs for the final fit. Tests use 1 trial, 1 epoch, and 2 folds on a tiny synthetic batch.


## Stage map the scaffold must create

| Stage | Reads | Writes |
| --- | --- | --- |
| 01 get data | `queries/images.sql` (not executed) | `data/mnist_ae_compression/raw_images.parquet`, `data/mnist_ae_compression/raw_images.npy` |
| 02 engineer | raw table and uint8 NHWC | `data/mnist_ae_compression/images.parquet`, `data/mnist_ae_compression/pixels.npy` in [-1, 1] NCHW |
| 03 split | `mnist_split` when both sides exist | `data/mnist_ae_compression/split.json` |
| 04 tune | train indices only | `reports/mnist_ae_compression/best_params.json`, `reports/mnist_ae_compression/mnist_ae.db`, Optuna HTML, latent-size PNG |
| 05 train | best params, full train split | `models/mnist_ae_compression/mnist_ae.pt`, `reports/mnist_ae_compression/plots/loss.png`, `reports/mnist_ae_compression/reconstructions/epoch_XXXX.png` |
| 06 eval | checkpoint, both splits | `reports/mnist_ae_compression/evaluation_train.json`, `reports/mnist_ae_compression/evaluation_test.json`, reconstruction PNGs |

Raw columns: `image_id`, `label`, `mnist_split`, and `label_name` mapped from the digit (`0` → `zero`, …, `9` → `nine`). Modeling features: `pixels` with shape `[1, 28, 28]`. Log columns: `label_name`, `mnist_split`, `pixel_mean`.

Package: `src/mnist_ae_compression`. Public class: `Autoencoder`. Env prefix for the checkpoint lookup: `MNIST_AE_`.


## Eval numbers and the reconstruction grid

Each of `reports/mnist_ae_compression/evaluation_train.json` and `reports/mnist_ae_compression/evaluation_test.json` contains `mse`, `mae`, `compression_ratio`, `latent_dim`, `n_train`, and `n_test`. MSE and MAE are scored on that split. The compression ratio is `in_features / latent_dim` from the checkpoint.

`reports/mnist_ae_compression/plots/reconstructions_train.png` and `reports/mnist_ae_compression/plots/reconstructions_test.png` show the same digits twice: the top row is the original, the bottom row is the reconstruction. Display uses the inverse of the stage-02 scale, clipped to [0, 1], in gray.

`reports/mnist_ae_compression/plots/loss.png` is MSE per epoch, with a `train` series and a `test` series.

`reports/mnist_ae_compression/reconstructions/epoch_XXXX.png` is that same original-over-reconstruction grid, written once per training epoch for a fixed prefix of the training images.


## After this notebook

Scaffold the repo from `dl-model-template-smbo` using the decisions above. Then add `notebooks/task_analysis.ipynb`, which:

1. Loads `reports/mnist_ae_compression/mnist_ae.db` and plots validation reconstruction MSE against latent size.
2. Loads `models/mnist_ae_compression/mnist_ae.pt` and draws original versus reconstruction for train and test samples.
3. Reads the two evaluation JSON files and states the compression ratio next to the error.

That notebook assumes `make all` has already been run. It does not start a new search.
